In [14]:
from util import get_all_energy_type, to_tabular_format
from cwarler import single_energy_record
from log_config import setup_logging

setup_logging("INFO")
energy_type = get_all_energy_type("Coal")
data = single_energy_record(energy_type['_id'])


2026-03-30 23:56:50 | INFO     | cwarler | [single_energy_record] START cid=9a59860aa4a9442e91d9aa01762e3284 indicator_type=0 dt_range=202508MM-202602MM
2026-03-30 23:56:50 | INFO     | cwarler | [1/4] Resolving region catalog ID for cid=9a59860aa4a9442e91d9aa01762e3284
2026-03-30 23:56:53 | INFO     | cwarler |       da_cid=a10dceae75d245008bf4b9a0e6fe1d55
2026-03-30 23:56:53 | INFO     | cwarler | [2/4] Fetching regions for da_cid=a10dceae75d245008bf4b9a0e6fe1d55
2026-03-30 23:56:55 | INFO     | cwarler |       31 regions: ['Beijing', 'Tianjin', 'Hebei', 'Shanxi', 'Inner Mongolia', 'Liaoning', 'Jilin', 'Heilongjiang', 'Shanghai', 'Jiangsu', 'Zhejiang', 'Anhui', 'Fujian', 'Jiangxi', 'Shandong', 'Henan', 'Hubei', 'Hunan', 'Guangdong', 'Guangxi', 'Hainan', 'Chongqing', 'Sichuan', 'Guizhou', 'Yunnan', 'Xizang', 'Shaanxi', 'Gansu', 'Qinghai', 'Ningxia', 'Xinjiang']
2026-03-30 23:56:55 | INFO     | cwarler | [3/4] Fetching indicators for cid=9a59860aa4a9442e91d9aa01762e3284
2026-03-30 23:5

In [11]:
# #  Convert Data to tabular formate
df = to_tabular_format(data.raw['data'],energy_type["_name"])
print(df.head(2))


       code      name energy_type  \
0  202602MM  Feb 2026        Coal   
1  202602MM  Feb 2026        Coal   

                                     i_showname  _name  \
0  Output of Coal, Current Period (10000 tons)   能源生产量   
1  Output of Coal, Current Period (10000 tons)   能源生产量   

                                             ek_dp kj1_name accuracy dp  \
0  124740211529|295834ae23bc45ed8124d53634d35e60_1       原煤        1  1   
1  124740211529|295834ae23bc45ed8124d53634d35e60_1       原煤        1  1   

        type  ... dead_line_value      areaCode   kj2           kj1  \
0  indicator  ...            None  110000000000  None  124740211529   
1  indicator  ...            None  120000000000  None  124740211529   

                          catalogid daterange_edate   kj3 i_mark  \
0  9a59860aa4a9442e91d9aa01762e3284            None  None   None   
1  9a59860aa4a9442e91d9aa01762e3284            None  None   None   

                                _id  du_name  
0  00930ccfd5944d81a1

In [0]:
catalog_name = "jeragm"
schema_name = "bronze_layer"
table_name = "ing_china_nbs_output_of_energy_products_current_period_actuals_monthly"


In [0]:
# Step 1: Create Schema
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog_name}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_name}")

In [0]:
# Step 2: Convert pandas -> Spark DataFrame
sdf = spark.createDataFrame(df)
sdf.printSchema()

In [8]:
# Step 3: Spark DataFrame -> Catlog
from pyspark.sql.functions import col

# -----------------------------
# Step 1: Define important columns (keep even if null)
# -----------------------------
important_cols = ["kj1_name", "kj2_name", "kj3_name"]

# -----------------------------
# Step 2: Convert important void columns → string
# -----------------------------
for field in sdf.schema.fields:
    if field.name in important_cols and field.dataType.typeName() == "void":
        sdf = sdf.withColumn(field.name, col(field.name).cast("string"))

# -----------------------------
# Step 3: Identify remaining void columns
# -----------------------------
void_cols = [
    field.name
    for field in sdf.schema.fields
    if field.dataType.typeName() == "void" and field.name not in important_cols
]

# -----------------------------
# Step 4: Drop unwanted void columns
# -----------------------------
if void_cols:
    print("Dropping void columns:", void_cols)
    sdf = sdf.drop(*void_cols)

# -----------------------------
# Step 5: Final schema validation (important!)
# -----------------------------
for field in sdf.schema.fields:
    if field.dataType.typeName() == "void":
        raise Exception(f"❌ Still found void column: {field.name}")

print("✅ Schema is clean. No void columns remaining.")

# -----------------------------
# Step 6: Save into Unity Catalog Delta table
# -----------------------------
sdf.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog_name}.{schema_name}.{table_name}")

print(f"✅ Saved successfully to {catalog_name}.{schema_name}.{table_name}")

ModuleNotFoundError: No module named 'pyspark'

In [0]:
sdf.head(5)

In [ ]:
# Finally Data 
from pyspark.sql.functions import col, trim

df = spark.table("jeragm.bronze_layer.ing_china_nbs_output_of_energy_products_current_period_actuals_monthly")

filtered_df = df.filter(
    col("value").isNotNull() & (trim(col("value")) != "")
)

filtered_df.select("code", "energy_type", "area", "value").show(5, truncate=False)